In [25]:
import pandas as pd

df = pd.read_csv("../data/hmif.csv")
df.head()

,question,answer
0,Apa itu HMIF?,HMIF adalah Himpunan Mahasiswa Informatika yan...
1,Apa tujuan utama HMIF?,Tujuan HMIF adalah mewadahi aspirasi serta pen...
2,HMIF berperan untuk siapa?,HMIF berperan untuk seluruh mahasiswa Program ...
3,Apakah HMIF merupakan organisasi resmi?,HMIF merupakan organisasi mahasiswa resmi yang...
4,Kapan HMIF didirikan?,HMIF didirikan pada tanggal 20 februaridan dir...


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  34 non-null     object
 1   answer    34 non-null     object
dtypes: object(2)
memory usage: 676.0+ bytes


In [27]:
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = stemmer.stem(text)
    return text

df['clean_question'] = df['question'].apply(preprocess)

df[['question', 'clean_question']].head()

,question,clean_question
0,Apa itu HMIF?,apa itu hmif
1,Apa tujuan utama HMIF?,apa tuju utama hmif
2,HMIF berperan untuk siapa?,hmif peran untuk siapa
3,Apakah HMIF merupakan organisasi resmi?,apakah hmif rupa organisasi resmi
4,Kapan HMIF didirikan?,kapan hmif diri


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_question'])

X.shape

(34, 76)

In [29]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def chatbot_response(user_input):
    user_input = preprocess(user_input)
    user_vec = vectorizer.transform([user_input])
    similarity = cosine_similarity(user_vec, X)
    idx = np.argmax(similarity)
    return df.iloc[idx]['answer']

In [30]:
chatbot_response("apa itu hmif")

'HMIF adalah Himpunan Mahasiswa Informatika yang menjadi wadah organisasi mahasiswa Program Studi Informatika.'

In [31]:
chatbot_response("hmif bergabung dengan permikomnas")

'Manfaatnya adalah memperluas jaringan pengalaman organisasi dan wawasan mahasiswa.'

In [32]:
import pickle

with open("../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("../models/hmif_chatbot_data.pkl", "wb") as f:
    pickle.dump(df, f)